# MGNO2d - the Multipole Graph Neural Operator

*Walkthrough notebook for `neural_operators/models/mgno.py`.*

GNO's radius truncation is cheap but short-sighted: reaching *far* points needs a big radius, which brings back the
$O(J^2)$ cost. The MGNO (paper Section 4.3) borrows from the **Fast Multipole Method**: let far-field interactions
happen on a **coarse grid**, where there are few enough points that a nearly fully-connected graph is cheap.

This reproduction uses two levels. One MGNO layer does four steps:

| Step | Direction | Radius | Captures |
|---|---|---|---|
| fine -> fine | full resolution | small | local / near-field interactions |
| **restrict** | fine -> coarse | medium | summarises local detail onto the coarse points |
| coarse -> coarse | coarse grid | huge (fully connected) | long-range / far-field interactions |
| **prolong** | coarse -> fine | medium | pushes the long-range result back to every fine point |

The three results are summed with the local linear term. All four steps reuse the same building block as GNO.

## Setup

The first cell makes the repository importable when this notebook is opened from `notebooks/source/`; the second holds the module's imports.

In [ ]:
# Make the repo root importable so `import neural_operators` works from notebooks/source/.
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "neural_operators").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from neural_operators.layers.kernel_nn import KernelMLP
from neural_operators.models.gno import build_radius_graph, make_grid_coords

## The shared building block

This is exactly the GNO message-passing update (kernel -> per-edge matrix -> multiply -> mean-aggregate),
but written to accept *different source and target point sets*, which restriction and prolongation need.

In [ ]:
def _message_pass(kernel_net, v_source, edge_index, edge_features, num_target):
    source, target = edge_index[0], edge_index[1]
    batch = v_source.shape[0]
    width = v_source.shape[-1]

    kappa = kernel_net(edge_features)               # (E, width, width)
    v_src = v_source[:, source, :]                   # (batch, E, width)
    messages = torch.einsum("eoi,bei->beo", kappa, v_src)  # (batch, E, width)

    agg = torch.zeros(batch, num_target, width, device=v_source.device, dtype=v_source.dtype)
    counts = torch.zeros(num_target, device=v_source.device, dtype=v_source.dtype)
    agg.index_add_(1, target, messages)
    counts.index_add_(0, target, torch.ones_like(target, dtype=v_source.dtype))
    agg = agg / counts.clamp(min=1.0).view(1, -1, 1)
    return agg

## One multipole layer

Four independent `KernelMLP`s - one per step - and a local linear term. `forward` chains the steps in the order
of the table above.

In [ ]:
class MGNOLayer(nn.Module):
    def __init__(self, width, coord_dim=2, hidden=32):
        super().__init__()
        self.fine_kernel = KernelMLP(2 * coord_dim, width, width, hidden=hidden)
        self.restrict_kernel = KernelMLP(2 * coord_dim, width, width, hidden=hidden)
        self.coarse_kernel = KernelMLP(2 * coord_dim, width, width, hidden=hidden)
        self.prolong_kernel = KernelMLP(2 * coord_dim, width, width, hidden=hidden)
        self.local_linear = nn.Linear(width, width)

    def forward(self, v_fine, graph):
        fine_msg = _message_pass(
            self.fine_kernel, v_fine[:, graph["fine_key_idx"], :],
            graph["fine_edge_index"], graph["fine_edge_features"], graph["n_fine"],
        )

        v_coarse = _message_pass(
            self.restrict_kernel, v_fine,
            graph["restrict_edge_index"], graph["restrict_edge_features"], graph["n_coarse"],
        )
        v_coarse = F.gelu(v_coarse)

        coarse_msg = _message_pass(
            self.coarse_kernel, v_coarse,
            graph["coarse_edge_index"], graph["coarse_edge_features"], graph["n_coarse"],
        )

        prolonged = _message_pass(
            self.prolong_kernel, coarse_msg,
            graph["prolong_edge_index"], graph["prolong_edge_features"], graph["n_fine"],
        )

        return fine_msg + prolonged + self.local_linear(v_fine)

## The whole network

`_build_graph` constructs every graph the layer needs (fine, restrict, coarse, prolong) once per resolution.
Note the coarse graph uses `radius=2.0`: larger than the unit square's diagonal, so every coarse point
connects to every other - cheap only because there are few of them (`coarse_stride=4` keeps one point in 16).

In [ ]:
class MGNO2d(nn.Module):
    def __init__(
        self,
        in_channels=3,
        out_channels=1,
        width=16,
        n_layers=4,
        fine_radius=0.08,
        restrict_radius=0.12,
        coarse_stride=4,
        n_fine_subsample=384,
        seed=0,
    ):
        super().__init__()
        self.width = width
        self.fine_radius = fine_radius
        self.restrict_radius = restrict_radius
        self.coarse_stride = coarse_stride
        self.n_fine_subsample = n_fine_subsample
        self.seed = seed

        self.lift = nn.Linear(in_channels, width)
        self.layers = nn.ModuleList([MGNOLayer(width) for _ in range(n_layers)])
        self.project1 = nn.Linear(width, 128)
        self.project2 = nn.Linear(128, out_channels)

        self._graph_cache = {}

    def _build_graph(self, size_x, size_y, device):
        fine_coords = make_grid_coords(size_x, size_y, device=device)
        n_fine = fine_coords.shape[0]

        g = torch.Generator(device="cpu").manual_seed(self.seed)
        n_sub = min(self.n_fine_subsample, n_fine)
        fine_key_idx = torch.randperm(n_fine, generator=g)[:n_sub].to(device)
        fine_key_coords = fine_coords[fine_key_idx]

        fine_edge_index = build_radius_graph(fine_coords, fine_key_coords, self.fine_radius)
        fine_edge_features = torch.cat(
            (fine_coords[fine_edge_index[1]], fine_key_coords[fine_edge_index[0]]), dim=-1
        )

        coarse_size_x = max(2, size_x // self.coarse_stride)
        coarse_size_y = max(2, size_y // self.coarse_stride)
        coarse_coords = make_grid_coords(coarse_size_x, coarse_size_y, device=device)
        n_coarse = coarse_coords.shape[0]

        # restrict: fine (source) -> coarse (target)
        restrict_edge_index = build_radius_graph(coarse_coords, fine_coords, self.restrict_radius)
        restrict_edge_features = torch.cat(
            (coarse_coords[restrict_edge_index[1]], fine_coords[restrict_edge_index[0]]), dim=-1
        )

        # coarse <-> coarse, large radius => effectively fully connected (captures long range cheaply)
        coarse_edge_index = build_radius_graph(coarse_coords, coarse_coords, radius=2.0)
        coarse_edge_features = torch.cat(
            (coarse_coords[coarse_edge_index[1]], coarse_coords[coarse_edge_index[0]]), dim=-1
        )

        # prolong: coarse (source) -> fine (target)
        prolong_edge_index = build_radius_graph(fine_coords, coarse_coords, self.restrict_radius)
        prolong_edge_features = torch.cat(
            (fine_coords[prolong_edge_index[1]], coarse_coords[prolong_edge_index[0]]), dim=-1
        )

        return {
            "fine_key_idx": fine_key_idx,
            "fine_edge_index": fine_edge_index,
            "fine_edge_features": fine_edge_features,
            "restrict_edge_index": restrict_edge_index,
            "restrict_edge_features": restrict_edge_features,
            "coarse_edge_index": coarse_edge_index,
            "coarse_edge_features": coarse_edge_features,
            "prolong_edge_index": prolong_edge_index,
            "prolong_edge_features": prolong_edge_features,
            "n_fine": n_fine,
            "n_coarse": n_coarse,
        }

    def _get_graph(self, size_x, size_y, device):
        key = (size_x, size_y, str(device))
        if key not in self._graph_cache:
            self._graph_cache[key] = self._build_graph(size_x, size_y, device)
        return self._graph_cache[key]

    def forward(self, a):
        if a.dim() == 3:
            a = a.unsqueeze(-1)
        batch, size_x, size_y = a.shape[0], a.shape[1], a.shape[2]
        device = a.device
        graph = self._get_graph(size_x, size_y, device)
        n_points = size_x * size_y

        fine_coords = make_grid_coords(size_x, size_y, device=device)
        a_flat = a.reshape(batch, n_points, -1)
        coords_b = fine_coords.unsqueeze(0).expand(batch, -1, -1)
        x = torch.cat((a_flat, coords_b), dim=-1)
        v = self.lift(x)

        for t, layer in enumerate(self.layers):
            v = layer(v, graph)
            if t < len(self.layers) - 1:
                v = F.gelu(v)

        out = F.gelu(self.project1(v))
        out = self.project2(out)
        return out.view(batch, size_x, size_y)

## Try it

Inspect the multi-level graph the model builds for a 32x32 input.

In [ ]:
from neural_operators.utils import count_params

model = MGNO2d()                          # saved result: 158,081 parameters
print("parameters:", count_params(model))

graph = model._get_graph(32, 32, "cpu")
print("fine points  :", graph["n_fine"])
print("coarse points:", graph["n_coarse"])
for name in ("fine", "restrict", "coarse", "prolong"):
    print(f"{name:>8} graph edges: {graph[name + '_edge_index'].shape[1]:>8,}")

print("output:", tuple(model(torch.randn(2, 32, 32)).shape))